# 12. Tools

Built-in tools, custom tools (`@tool`, `StructuredTool`, `BaseTool`) and toolkits.

> Setup: `pip install -r requirements.txt` and copy `.env.example` to `.env` first.

In [ ]:
from dotenv import load_dotenv

load_dotenv()  # reads OPENAI_API_KEY (and others) from the .env file in the repo root

## Built-in Tool - DuckDuckGo Search

In [ ]:
from langchain_community.tools import DuckDuckGoSearchRun

search_tool = DuckDuckGoSearchRun()

results = search_tool.invoke('top news in india today')

print(results)

In [ ]:
print(search_tool.name)
print(search_tool.description)
print(search_tool.args)

## Built-in Tool - Shell Tool

> **Warning:** `ShellTool` executes real commands on your machine. Never expose it to untrusted input or an unsupervised agent.

In [ ]:
from langchain_community.tools import ShellTool

shell_tool = ShellTool()

results = shell_tool.invoke('ls')

print(results)

## Custom Tools

In [ ]:
from langchain_core.tools import tool

In [ ]:
# Step 1 - create a function

def multiply(a, b):
    """Multiply two numbers"""
    return a*b

In [ ]:
# Step 2 - add type hints

def multiply(a: int, b:int) -> int:
    """Multiply two numbers"""
    return a*b

In [ ]:
# Step 3 - add tool decorator

@tool
def multiply(a: int, b:int) -> int:
    """Multiply two numbers"""
    return a*b

In [ ]:
result = multiply.invoke({"a":3, "b":5})

In [ ]:
print(result)

In [ ]:
print(multiply.name)
print(multiply.description)
print(multiply.args)

In [ ]:
print(multiply.args_schema.model_json_schema())

## Method 2 - Using StructuredTool

In [ ]:
from langchain_core.tools import StructuredTool
from pydantic import BaseModel, Field

In [ ]:
class MultiplyInput(BaseModel):
    a: int = Field(..., description="The first number to multiply")
    b: int = Field(..., description="The second number to multiply")

In [ ]:
def multiply_func(a: int, b: int) -> int:
    return a * b

In [ ]:
multiply_tool = StructuredTool.from_function(
    func=multiply_func,
    name="multiply",
    description="Multiply two numbers",
    args_schema=MultiplyInput
)

In [ ]:
result = multiply_tool.invoke({'a':3, 'b':3})

print(result)
print(multiply_tool.name)
print(multiply_tool.description)
print(multiply_tool.args)

## Method 3 - Using BaseTool Class

In [ ]:
from langchain_core.tools import BaseTool
from typing import Type

In [ ]:
# arg schema using pydantic

class MultiplyInput(BaseModel):
    a: int = Field(..., description="The first number to multiply")
    b: int = Field(..., description="The second number to multiply")

In [ ]:
class MultiplyTool(BaseTool):
    name: str = "multiply"
    description: str = "Multiply two numbers"

    args_schema: Type[BaseModel] = MultiplyInput

    def _run(self, a: int, b: int) -> int:
        return a * b

In [ ]:
multiply_tool = MultiplyTool()

In [ ]:
result = multiply_tool.invoke({'a':3, 'b':3})

print(result)
print(multiply_tool.name)
print(multiply_tool.description)

print(multiply_tool.args)

## Toolkit

In [ ]:
from langchain_core.tools import tool

# Custom tools
@tool
def add(a: int, b: int) -> int:
    """Add two numbers"""
    return a + b

@tool
def multiply(a: int, b: int) -> int:
    """Multiply two numbers"""
    return a * b

In [ ]:
class MathToolkit:
    def get_tools(self):
        return [add, multiply]

In [ ]:
toolkit = MathToolkit()
tools = toolkit.get_tools()

for t in tools:
    print(t.name, "=>", t.description)